# Figures of "Pain-evoked neural activity scales with subjective pain across individuals"

This notebook redraws every figure and rebuilds the tables reported in the paper and its supplement. It uses
the shared group-level data in `../data` and does not need any single-participant image.

The analysis code is in the `painpred` package next to this notebook. Each section below calls one step and
shows the result. Steps whose output already exists are skipped; set `OVERWRITE = True` to recompute.

If you have the raw study files, set `STUDY_FOLDER` and the notebook also reruns the image-based steps
(permutation maps, LASSO-PCR, Hoeppli registration). Those steps take hours.

## Setup

In [ ]:
import sys
from pathlib import Path

from IPython.display import Image, display

sys.path.insert(0, str(Path.cwd()))   # lets the notebook find `painpred` without installing it

from painpred.config import Project
from painpred.pipeline import run_steps

DATA = Path("../data")
RESULTS = Path("../results")
STUDY_FOLDER = None       # e.g. Path("/path/to/study"); only for people with the raw images
OVERWRITE = False

if STUDY_FOLDER:
    project = Project(input_root=STUDY_FOLDER, results_root=RESULTS)
    print("Raw study files:", project.input_root)
else:
    project = Project(data_root=DATA, results_root=RESULTS)
    print("Shared data:", project.data_root)
print("Results go to:", project.results_root)


def show(name):
    path = project.figures / f"{name}.png"
    display(Image(filename=str(path), width=900)) if path.exists() else print(f"{path} not found")

## Figure 1: ratings and calibration

Mean pain ratings for 46 and 48 °C at arm and leg, how much each person's ratings vary across the 12
repetitions, and the pre-experiment calibration temperatures.

In [ ]:
run_steps(project, ["figure1"], OVERWRITE)
show("Figure1")

## Voxelwise correlations (raw study files only)

Spearman correlation between the heat contrast and the pain rating at every voxel, for four contrasts and four
ratings. Significance comes from 10,000 permutations of the ratings: the maximum |z| gives
voxelwise FWE, the largest cluster (|z| ≥ 3.1) gives cluster-extent FWE. With the shared data this step is
skipped, the finished maps are in `data/derivatives/painpred/figure2/maps`.

In [ ]:
run_steps(project, ["hoeppli_mask", "massuni"], OVERWRITE)

## Figure 2: where heat-evoked activity tracks pain

Maps for the 48 °C arm contrast and 48 °C arm rating, the activity-rating relationship at four peaks, a table of
local peaks, and the overlap with the same-study 48 °C map and the independent Hoeppli map.

In [ ]:
run_steps(project, ["figure2_peaks", "figure2_overlap", "figure2"], OVERWRITE)
show("Figure2")

## Figure 3: agreement across the four ratings, and the NPS

Number of rating analyses (0 to 4) in which each voxel survives voxelwise FWE, the atlas regions that survive in
all four, and the relation between NPS expression and mean pain.

In [ ]:
run_steps(project, ["figure3_overlap", "figure3"], OVERWRITE)
show("Figure3")

## Figure 4: LASSO-PCR signature (fitting needs the raw study files)

Nested cross-validation (5 × 5 outer folds; principal components and penalty chosen in the inner folds), the final
model, and the comparison with the NPS using the same folds. The fitted weights and cross-validation tables are
part of the shared data.

In [ ]:
run_steps(project, ["lasso_pcr", "figure4"], OVERWRITE)
show("Figure4")

## Figure 5: pain versus non-pain contrasts

NPS and signature expression for the four heat contrasts, sound and the checkerboard, plus group T maps of the
two control contrasts.

In [ ]:
run_steps(project, ["figure5"], OVERWRITE)
show("Figure5")

## Figure 6: combined models

Three linear models (calibration alone, NPS alone, both) compared by repeated 5-fold cross-validation, a
participant bootstrap and a prediction-swap permutation test.

In [ ]:
run_steps(project, ["figure6"], OVERWRITE)
show("Figure6")

## Supplement

* **Figure S1**: simulation of the voxelwise correlation procedure (needs no data).
* **Figure S2**: maps of the three other contrasts.
* Warmth against implicit baseline as reference condition.

In [ ]:
run_steps(project, ["figure_s1", "figure_s2", "supplement"], OVERWRITE)
show("FigureS1")
show("FigureS2")

## Check against an earlier run

To confirm that a new results folder reproduces an older one, compare all tables and maps:

In [ ]:
from painpred.compare import compare_results

OLD_RESULTS = None    # path to the results folder to compare with
if OLD_RESULTS:
    compare_results(OLD_RESULTS, project.results_root)